# TRACK-FA Raw Combination and Drop3POMs Missingness Audit

This notebook is table-only. It checks missing entities, missing columns, and missing visits at two stages:

1. Direct raw REDCap + raw MasterFile entry join by `participant_id` and `visit`.
2. Final modelling dataset `trackfa_pairs_drop3poms.csv`.

`trackfa_pairs_drop3poms.csv` is treated as the correct paired modelling CSV. In that file, `feature_baseline` is the V1 or V2 value, `feature_followup` is the V2 or V3 value, and `delta_feature` is either V2-V1 or V3-V2, depending on `patient_id` suffix `V1V2` or `V2V3`.


In [22]:
from __future__ import annotations

import ast
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    for path in (start.resolve(), *start.resolve().parents):
        if (path / "src").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Could not find the project root containing src/ and data/.")

REPO_ROOT = find_project_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data.trackfa import raw_direct_combination_audit, run_merge

PAIRS_PATH = REPO_ROOT / "data" / "processed" / "trackfa_pairs_drop3poms.csv"


DISPLAY_COLUMN_NAMES = {
    "source": "Source",
    "raw_rows": "Raw Rows",
    "raw_columns": "Raw Columns",
    "visit_rows_used_for_direct_join": "Visit Rows Used for Direct Join",
    "unique_participants": "Unique Participants",
    "source_presence": "Source Presence",
    "n_rows": "Number of Rows",
    "visit": "Visit",
    "source_block": "Source Block",
    "n_columns": "Number of Columns",
    "mean_missing_pct": "Mean Missing %",
    "max_missing_pct": "Maximum Missing %",
    "column": "Column",
    "missing_pct": "Missing %",
    "n_missing": "Number Missing",
    "patient_id": "Patient ID",
    "pair_type": "Pair Type",
    "subject": "Subject ID",
    "n_pair_rows": "Number of Pair Rows",
    "n_unique_subjects": "Number of Unique Subjects",
    "n_v1v2_rows": "Number of Visit 1 to Visit 2 Rows",
    "n_v2v3_rows": "Number of Visit 2 to Visit 3 Rows",
    "n_subjects_with_both_intervals": "Number of Subjects with Both Intervals",
    "n_subjects_with_one_interval": "Number of Subjects with One Interval",
    "observed_intervals": "Observed Intervals",
    "missing_v1v2": "Missing Visit 1 to Visit 2 Interval",
    "missing_v2v3": "Missing Visit 2 to Visit 3 Interval",
    "feature_or_column": "Feature or Column",
    "reason": "Reason",
    "removed_pair_column": "Removed Pair Column",
    "notebook": "Notebook",
    "uses_trackfa_pairs_drop3poms": "Uses Required Drop 3 POMs Pair CSV",
    "has_fallback_to_trackfa_pairs_csv": "Has Fallback to Strict Pair CSV",
    "uses_trackfa_pairs_csv_directly": "Uses Strict Pair CSV Directly",
    "status": "Status",
    "number_of_subjects": "Number of Subjects",
    "subjects_with_all_3_visit_values": "Subjects with All 3 Visit Values",
    "subjects_missing_at_least_1_visit_value": "Subjects Missing at Least 1 Visit Value",
    "has_visit_1_values": "Has Visit 1 Values",
    "has_visit_2_values": "Has Visit 2 Values",
    "has_visit_3_values": "Has Visit 3 Values",
    "has_all_3_visit_values": "Has All 3 Visit Values",
    "missing_visit_values": "Missing Visit Values",
    "number_of_subjects_with_visit_1_values": "Number of Subjects with Visit 1 Values",
    "number_of_subjects_with_visit_2_values": "Number of Subjects with Visit 2 Values",
    "number_of_subjects_with_visit_3_values": "Number of Subjects with Visit 3 Values",
    "number_of_subjects_with_all_3_visit_values": "Number of Subjects with All 3 Visit Values",
    "visit_pair": "Visit Pair",
}

DISPLAY_VALUE_REPLACEMENTS = {
    "both": "Clinical and Imaging",
    "left_only": "Clinical Only",
    "right_only": "Imaging Only",
    "clinical_raw": "Raw Clinical Columns",
    "imaging_raw": "Raw Imaging Columns",
    "V1V2": "Visit 1 to Visit 2",
    "V2V3": "Visit 2 to Visit 3",
    "OK": "Pass",
    "CHECK": "Check",
}


def display_table(frame: pd.DataFrame):
    """Display audit tables with full labels and percentage units."""
    out = frame.copy()
    for col in out.columns:
        if col.endswith("pct") or col.endswith("_pct") or col in {"missing_pct", "mean_missing_pct", "max_missing_pct"}:
            out[col] = pd.to_numeric(out[col], errors="coerce") * 100.0
    out = out.replace(DISPLAY_VALUE_REPLACEMENTS)
    out = out.rename(columns={k: v for k, v in DISPLAY_COLUMN_NAMES.items() if k in out.columns})
    percent_cols = [c for c in out.columns if c.endswith("%")]
    if percent_cols:
        styled = out.style.format({c: "{:.2f}%" for c in percent_cols}, na_rep="")
        display(styled)
    else:
        display(out)



def keep_clinical_total_missing_columns(frame: pd.DataFrame) -> pd.DataFrame:
    """For raw clinical columns, display totals plus core demographics/genetics."""
    if "source_block" not in frame.columns or "column" not in frame.columns:
        return frame
    source = frame["source_block"].astype(str)
    column = frame["column"].astype(str)
    is_raw_clinical = source.eq("clinical_raw") | column.str.startswith("clinical__")
    clinical_name = column.str.replace(r"^clinical__", "", regex=True)
    core_clinical = {"age", "gender", "gaa_1"}
    keep_clinical = clinical_name.str.endswith("_total") | clinical_name.isin(core_clinical)
    keep = ~is_raw_clinical | keep_clinical
    return frame.loc[keep].copy()


## 1. Regenerate Processed TRACK-FA Pair Dataset

This cell runs the actual merge pipeline from raw REDCap and MasterFile inputs, saves the standard pair file, and exports `trackfa_pairs_drop3poms.csv` for modelling.


In [23]:
# Regenerate processed TRACK-FA outputs from the raw sources.
# This exports data/processed/trackfa_pairs_drop3poms.csv when save_drop3_poms_variant=True.
long_df, _strict_pairs_df = run_merge(
    save=True,
    verbose=True,
    drop_incomplete_clinical_pairs=True,
    require_complete_imaging_sheets=True,
    save_drop3_poms_variant=True,
)

if not PAIRS_PATH.exists():
    raise FileNotFoundError(f"Expected exported modelling dataset not found: {PAIRS_PATH}")

pairs_df = pd.read_csv(PAIRS_PATH)
print(f"Exported and loaded required modelling CSV: {PAIRS_PATH}")
print(f"Shape: {pairs_df.shape[0]} rows x {pairs_df.shape[1]} columns")


[trackfa_pairs] dropped 2 rows with missing clinical values (before=298, after=296). Removed by pair: {'V1V2': 1, 'V2V3': 1}
[saved] /Users/robertwang/Documents/New_project/biomarkers/data/processed/trackfa_pairs_drop3poms.csv (207 rows × 455 cols); pair counts: {'V1V2': 108, 'V2V3': 99}
[trackfa_pairs] dropped 222 rows failing strict sheet completeness (before=296, after=74). Removed by pair: {'V1V2': 114, 'V2V3': 108}
[trackfa_pairs] failed-sheet counts (rows where this condition fails): POMs_complete_baseline=147, POMs_complete_followup=167, BrainSpineMorph_complete_baseline=29, BrainSpineMorph_complete_followup=26, BrainDTI_complete_baseline=42, BrainDTI_complete_followup=40
[trackfa_pairs] top missing columns among removed rows:
delta_tNAA_myo_Ins       0.653153
delta_DN_suscept         0.621622
delta_DN_vol             0.621622
tNAA_myo_Ins_followup    0.527027
DN_suscept_baseline      0.436937
DN_vol_baseline          0.436937
DN_suscept_followup      0.432432
DN_vol_followup   

## 2. Direct Raw Entry Join Missingness


In [24]:
raw_audit = raw_direct_combination_audit()
raw_combined = raw_audit["combined"]

print("Raw source sizes and direct outer-join size")
display_table(raw_audit["source_summary"])

print("Missing entities/rows by source presence")
display_table(raw_audit["presence_summary"])

print("Missing visits by source presence")
display_table(raw_audit["visit_presence"])

print("Missing columns by raw source block")
display_table(raw_audit["block_summary"])

raw_missing_columns = keep_clinical_total_missing_columns(raw_audit["missingness"].query("missing_pct > 0").copy())
print(f"Columns with any missingness after direct raw join: {len(raw_missing_columns)}")
display_table(raw_missing_columns)

raw_missing_by_visit = keep_clinical_total_missing_columns(raw_audit["missingness_by_visit"].query("missing_pct > 0").copy())
print(f"Column x visit missingness rows after direct raw join: {len(raw_missing_by_visit)}")
display_table(raw_missing_by_visit.sort_values(["missing_pct", "visit", "column"], ascending=[False, True, True], kind="mergesort"))


Raw source sizes and direct outer-join size


,Source,Raw Rows,Raw Columns,Visit Rows Used for Direct Join,Unique Participants
0,REDCap raw export,1076,483,807,269
1,Imaging MasterFile all sheets,793,157,747,269
2,Direct outer join,807,639,807,269


Missing entities/rows by source presence


/var/folders/2y/d2x11n9s4sbc3j2svdb5qdb00000gn/T/ipykernel_21235/2226729835.py:98: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  out = out.replace(DISPLAY_VALUE_REPLACEMENTS)


,Source Presence,Number of Rows
0,Clinical and Imaging,747
1,Clinical Only,60
2,Imaging Only,0


...

,Source Block,Number of Columns,Mean Missing %,Maximum Missing %
0,Raw Clinical Columns,481,80.55%,100.00%
1,Raw Imaging Columns,155,20.65%,39.65%


Columns with any missingness after direct raw join: 159


,Column,Missing %,Source Block
0,clinical__age,100.00%,Raw Clinical Columns
8,clinical__gaa_1,100.00%,Raw Clinical Columns
10,clinical__gender,100.00%,Raw Clinical Columns
476,clinical__upenn_fxn_total,66.67%,Raw Clinical Columns
480,imaging__tNAA_myo_Ins,39.65%,Raw Imaging Columns
481,imaging__DN_suscept,36.93%,Raw Imaging Columns
482,imaging__DN_vol,36.93%,Raw Imaging Columns
483,imaging__AD_CST,23.42%,Raw Imaging Columns
484,imaging__AD_ICP,23.42%,Raw Imaging Columns
485,imaging__AD_MCP,23.42%,Raw Imaging Columns


Column x visit missingness rows after direct raw join: 476


,Visit,Column,Missing %,Source Block
15,1,clinical__age,100.00%,Raw Clinical Columns
72,1,clinical__gaa_1,100.00%,Raw Clinical Columns
21,1,clinical__gender,100.00%,Raw Clinical Columns
16,2,clinical__age,100.00%,Raw Clinical Columns
73,2,clinical__gaa_1,100.00%,Raw Clinical Columns
22,2,clinical__gender,100.00%,Raw Clinical Columns
1393,2,clinical__upenn_fxn_total,100.00%,Raw Clinical Columns
17,3,clinical__age,100.00%,Raw Clinical Columns
74,3,clinical__gaa_1,100.00%,Raw Clinical Columns
23,3,clinical__gender,100.00%,Raw Clinical Columns


## 3. Final Entity, Column, and Visit Audit


In [25]:
pair_meta = pairs_df["patient_id"].astype(str).str.extract(r"(?P<subject>.+)_(?P<pair_type>V1V2|V2V3)$")

value_columns = [c for c in pairs_df.columns if c != "patient_id"]
missing_feature_entities = (
    pairs_df[value_columns]
    .isna()
    .sum()
    .rename("n_missing")
    .reset_index()
    .rename(columns={"index": "column"})
)
missing_feature_entities["missing_pct"] = missing_feature_entities["n_missing"] / len(pairs_df)
missing_feature_entities = missing_feature_entities[missing_feature_entities["n_missing"] > 0].sort_values(
    ["n_missing", "column"], ascending=[False, True], kind="mergesort"
)

subject_visit_availability = pair_meta.groupby("subject")["pair_type"].apply(lambda s: set(s)).reset_index(name="observed_intervals")
subject_visit_availability["has_visit_1_values"] = subject_visit_availability["observed_intervals"].apply(lambda s: "V1V2" in s)
subject_visit_availability["has_visit_2_values"] = subject_visit_availability["observed_intervals"].apply(lambda s: bool({"V1V2", "V2V3"} & s))
subject_visit_availability["has_visit_3_values"] = subject_visit_availability["observed_intervals"].apply(lambda s: "V2V3" in s)
subject_visit_availability["has_all_3_visit_values"] = subject_visit_availability[
    ["has_visit_1_values", "has_visit_2_values", "has_visit_3_values"]
].all(axis=1)

visit_count_summary = pd.DataFrame({
    "number_of_subjects_with_visit_1_values": [int(subject_visit_availability["has_visit_1_values"].sum())],
    "number_of_subjects_with_visit_2_values": [int(subject_visit_availability["has_visit_2_values"].sum())],
    "number_of_subjects_with_visit_3_values": [int(subject_visit_availability["has_visit_3_values"].sum())],
    "number_of_subjects_with_all_3_visit_values": [int(subject_visit_availability["has_all_3_visit_values"].sum())],
})

missing_visit_subjects = subject_visit_availability.loc[
    ~subject_visit_availability["has_all_3_visit_values"],
    ["subject", "observed_intervals"],
].copy()
missing_visit_subjects["visit_pair"] = missing_visit_subjects["observed_intervals"].apply(
    lambda s: ", ".join(sorted(s))
)
missing_visit_subjects = missing_visit_subjects[["subject", "visit_pair"]].sort_values(
    "subject", kind="mergesort"
).reset_index(drop=True)

print("Number of subjects with Visit 1, Visit 2, Visit 3, and all 3 visit values")
display_table(visit_count_summary)

print(f"Subjects missing at least one visit value: {len(missing_visit_subjects)}")
display_table(missing_visit_subjects)


Number of subjects with Visit 1, Visit 2, Visit 3, and all 3 visit values


,Number of Subjects with Visit 1 Values,Number of Subjects with Visit 2 Values,Number of Subjects with Visit 3 Values,Number of Subjects with All 3 Visit Values
0,108,117,99,90


Subjects missing at least one visit value: 27


,Subject ID,Visit Pair
0,AAN003,Visit 1 to Visit 2
1,AAN037,Visit 1 to Visit 2
2,AAN042,Visit 1 to Visit 2
3,AAN045,Visit 2 to Visit 3
4,CHP014,Visit 1 to Visit 2
5,CHP020,Visit 2 to Visit 3
6,CHP030,Visit 1 to Visit 2
7,CHP040,Visit 1 to Visit 2
8,CHP047,Visit 1 to Visit 2
9,CHP051,Visit 2 to Visit 3


## 6. Longitudinal Effective Sample Size Audit

This block answers the longitudinal sample-size checklist using the actual training dataset, `trackfa_pairs_drop3poms.csv`. It does not infer paired N from marginal counts and does not modify the modelling pipeline.


In [ ]:
# DATA AUDIT ONLY: effective longitudinal sample sizes from the actual training CSV.


training_pairs = pairs_df.copy()
parsed = training_pairs["patient_id"].astype(str).str.extract(r"(?P<subject>.+)_(?P<pair_type>V1V2|V2V3)$")
if parsed.isna().any().any():
    bad = training_pairs.loc[parsed.isna().any(axis=1), "patient_id"].head(10).tolist()
    raise ValueError(f"Could not parse patient_id as subject_interval for examples: {bad}")
training_pairs = training_pairs.assign(subject_id=parsed["subject"], pair_type=parsed["pair_type"])

subject_id_col = "subject_id parsed from patient_id before suffix _V1V2/_V2V3"
visit_variable = "pair_type parsed from patient_id suffix"
visit_mapping = pd.DataFrame([
    {"encoded_value": "V1V2", "baseline_visit": "V1", "followup_visit": "V2", "duration": "~1 year"},
    {"encoded_value": "V2V3", "baseline_visit": "V2", "followup_visit": "V3", "duration": "~1 year"},
])

# Duplicate pair records in the actual training dataset.
duplicate_pairs = (
    training_pairs.groupby(["subject_id", "pair_type"], dropna=False)
    .size()
    .reset_index(name="n_rows")
    .query("n_rows > 1")
    .reset_index(drop=True)
)

# Convert pair rows into subject-level visit availability. V2 is observed if either adjacent interval is present.
subject_sets = training_pairs.groupby("subject_id")["pair_type"].apply(lambda s: set(s)).reset_index(name="observed_pair_types")
subject_sets["has_V1"] = subject_sets["observed_pair_types"].apply(lambda s: "V1V2" in s)
subject_sets["has_V2"] = subject_sets["observed_pair_types"].apply(lambda s: bool({"V1V2", "V2V3"} & s))
subject_sets["has_V3"] = subject_sets["observed_pair_types"].apply(lambda s: "V2V3" in s)
subject_sets["pattern"] = subject_sets.apply(
    lambda r: f"{int(r['has_V1'])}{int(r['has_V2'])}{int(r['has_V3'])}", axis=1
)

n_unique_subjects = int(subject_sets["subject_id"].nunique())
n_v1 = int(subject_sets["has_V1"].sum())
n_v2 = int(subject_sets["has_V2"].sum())
n_v3 = int(subject_sets["has_V3"].sum())
n12 = int((subject_sets["has_V1"] & subject_sets["has_V2"]).sum())
n23 = int((subject_sets["has_V2"] & subject_sets["has_V3"]).sum())
n13 = int((subject_sets["has_V1"] & subject_sets["has_V3"]).sum())
n123 = int((subject_sets["has_V1"] & subject_sets["has_V2"] & subject_sets["has_V3"]).sum())

marginal_visit_counts = pd.DataFrame([
    {"Visit": "V1", "Unique subjects": n_v1},
    {"Visit": "V2", "Unique subjects": n_v2},
    {"Visit": "V3", "Unique subjects": n_v3},
])
pair_availability = pd.DataFrame([
    {"Interval": "V1->V2", "Duration": "~1 year", "Visit-available N": n12},
    {"Interval": "V2->V3", "Duration": "~1 year", "Visit-available N": n23},
    {"Interval": "V1->V3", "Duration": "~2 years", "Visit-available N": n13},
])
pattern_meaning = {
    "111": "V1,V2,V3",
    "110": "V1,V2 only",
    "101": "V1,V3 only",
    "011": "V2,V3 only",
    "100": "V1 only",
    "010": "V2 only",
    "001": "V3 only",
}
visit_pattern_counts = (
    subject_sets["pattern"].value_counts().rename_axis("Pattern").reset_index(name="N")
)
visit_pattern_counts["Meaning"] = visit_pattern_counts["Pattern"].map(pattern_meaning)
visit_pattern_counts = (
    pd.DataFrame({"Pattern": list(pattern_meaning), "Meaning": list(pattern_meaning.values())})
    .merge(visit_pattern_counts[["Pattern", "N"]], on="Pattern", how="left")
    .fillna({"N": 0})
)
visit_pattern_counts["N"] = visit_pattern_counts["N"].astype(int)
pattern_sum_check = int(visit_pattern_counts["N"].sum())

# Model-usable counts: drop3poms has no missing model columns, so adjacent rows are model-usable by construction.
model_cols = [c for c in training_pairs.columns if c not in {"patient_id", "subject_id", "pair_type"}]
row_model_usable = ~training_pairs[model_cols].isna().any(axis=1)
usable_by_pair = training_pairs.loc[row_model_usable].groupby("pair_type")["subject_id"].nunique()
usable_subject_sets = training_pairs.loc[row_model_usable].groupby("subject_id")["pair_type"].apply(lambda s: set(s))
n12_model = int(usable_by_pair.get("V1V2", 0))
n23_model = int(usable_by_pair.get("V2V3", 0))
n13_model = int(usable_subject_sets.apply(lambda s: {"V1V2", "V2V3"}.issubset(s)).sum()) if len(usable_subject_sets) else 0
model_usable_counts = pd.DataFrame([
    {"Interval": "V1->V2", "Model-usable N": n12_model, "Eligibility rule": "drop3poms row exists and has no missing model columns"},
    {"Interval": "V2->V3", "Model-usable N": n23_model, "Eligibility rule": "drop3poms row exists and has no missing model columns"},
    {"Interval": "V1->V3", "Model-usable N": n13_model, "Eligibility rule": "subject has both V1V2 and V2V3 usable rows"},
])

# Clinical comparator counts from the same actual training CSV.
def _clinical_count(delta_col, required_pair_types):
    if delta_col not in training_pairs.columns:
        return 0
    subset = training_pairs[training_pairs["pair_type"].isin(required_pair_types)]
    if required_pair_types == {"V1V2", "V2V3"}:
        valid = subset.dropna(subset=[delta_col]).groupby("subject_id")["pair_type"].apply(lambda s: set(s))
        return int(valid.apply(lambda s: {"V1V2", "V2V3"}.issubset(s)).sum()) if len(valid) else 0
    return int(subset.dropna(subset=[delta_col])["subject_id"].nunique())

clinical_pair_counts = pd.DataFrame([
    {
        "Measure": "MRI visit availability",
        "V1->V2 N": n12,
        "V2->V3 N": n23,
        "V1->V3 N": n13,
    },
    {
        "Measure": "MRI model-usable",
        "V1->V2 N": n12_model,
        "V2->V3 N": n23_model,
        "V1->V3 N": n13_model,
    },
    {
        "Measure": "FARS/mFARS",
        "V1->V2 N": _clinical_count("delta_mfars_total", {"V1V2"}),
        "V2->V3 N": _clinical_count("delta_mfars_total", {"V2V3"}),
        "V1->V3 N": _clinical_count("delta_mfars_total", {"V1V2", "V2V3"}),
    },
    {
        "Measure": "SARA",
        "V1->V2 N": _clinical_count("delta_sara_total", {"V1V2"}),
        "V2->V3 N": _clinical_count("delta_sara_total", {"V2V3"}),
        "V1->V3 N": _clinical_count("delta_sara_total", {"V1V2", "V2V3"}),
    },
])

only_v12 = int(subject_sets["observed_pair_types"].apply(lambda s: s == {"V1V2"}).sum())
only_v23 = int(subject_sets["observed_pair_types"].apply(lambda s: s == {"V2V3"}).sum())
both_annual = int(subject_sets["observed_pair_types"].apply(lambda s: {"V1V2", "V2V3"}.issubset(s)).sum())
annual_overlap = pd.DataFrame([
    {"Category": "Subjects contributing only V1->V2", "N": only_v12},
    {"Category": "Subjects contributing only V2->V3", "N": only_v23},
    {"Category": "Subjects contributing both annual intervals", "N": both_annual},
])

n_diff = abs(n12_model - n23_model)
overlap_fraction = both_annual / n_unique_subjects if n_unique_subjects else np.nan
feasibility = pd.DataFrame([
    {"Question": "Is V1->V2 sufficiently populated for primary 12-month evaluation?", "Answer": f"Yes for this dataset: N12_model={n12_model}."},
    {"Question": "Is V2->V3 sufficiently populated for temporal replication?", "Answer": f"Yes, but smaller than V1->V2: N23_model={n23_model}."},
    {"Question": "How much overlap exists?", "Answer": f"{both_annual} subjects contribute to both annual intervals ({overlap_fraction:.1%} of unique training subjects)."},
    {"Question": "How much information is lost by requiring complete V1/V2/V3 cases?", "Answer": f"Complete annual-interval subjects N123={n123}; this excludes {n_unique_subjects - n123} of {n_unique_subjects} subjects."},
    {"Question": "Should complete-case-only modelling be avoided?", "Answer": "Avoid complete-case-only modelling if the goal is to use all adjacent interval information; complete-case restriction would drop one-interval subjects."},
    {"Question": "Are N12 and N23 similar enough for equal mean annual dz?", "Answer": f"N12_model={n12_model}, N23_model={n23_model}, difference={n_diff}. Equal weighting may be acceptable as a simple summary, but report interval-specific d_z and overlap because intervals are not independent."},
    {"Question": "Does tuning strategy need modification?", "Answer": "Do not tune on marginal visit counts. Use paired/model-usable counts and report V1->V2 and V2->V3 separately; avoid treating N12+N23 as independent patient N."},
    {"Question": "Are duplicate or missing-data issues likely to change effective model-training N?", "Answer": f"Duplicate subject-pair rows={len(duplicate_pairs)}; missing model columns in drop3poms={int(training_pairs[model_cols].isna().sum().sum())}. Effective N is governed by pair availability."},
])



print(f"N12 = {n12}")
print(f"N23 = {n23}")
print(f"N13 = {n13}")
print(f"N123 = {n123}")
print("\nIdentifier and visit encoding")
display(pd.DataFrame([{ "Subject ID column": subject_id_col, "Total unique subjects": n_unique_subjects, "Visit variable": visit_variable }]))
display(visit_mapping)
print("\nMarginal unique-subject visit counts")
display(marginal_visit_counts)
print("\nCritical paired sample sizes")
display(pair_availability)
print("\nComplete three-visit N")
display(pd.DataFrame([{ "N123 subjects with V1,V2,V3": n123 }]))
print("\nVisit-pattern counts")
display(visit_pattern_counts)
print(f"Pattern sum check: {pattern_sum_check} == {n_unique_subjects}")
print("\nDuplicate subject-pair records")
display(duplicate_pairs)
print("\nMRI model-usable counts")
display(model_usable_counts)
print("\nClinical comparator pair counts")
display(clinical_pair_counts)
print("\nAnnual interval overlap")
display(annual_overlap)
print("\nFollow-up duration")
display(pd.DataFrame([{ "Status": "No actual visit dates are retained in trackfa_pairs_drop3poms.csv; use nominal V1->V2 ~1y, V2->V3 ~1y, V1->V3 ~2y unless raw date audit is added separately." }]))
print("\nFeasibility and implications")
display(feasibility)

